# Curva de precio horario a largo plazo · TFM Energía UCM

Una sola función, `curva(desde, hasta)`, que devuelve precio horario para **cualquier**
rango y dice de dónde sale cada día:

| origen | de dónde | cuándo se usa |
|---|---|---|
| `historico` | `spot_price`, el PMD publicado | hasta ayer |
| `modelo` | tabla `predictions`, el ensemble | los días que se hayan predicho |
| `simulado` | generado, con banda P10-P90 | más allá |

## Por qué los modelos de D+1 no sirven para esto

Los ocho modelos entrenados reciben **los 7 días previos observados** más las previsiones de
D+1. Para llegar a 2046 habría que realimentar sus propias predicciones 7.300 veces: el error
se compone y en pocos días la serie se aplana a la media. Y los exógenos de D+1 —demanda
prevista, eólica, gas, CO2— no existen para dentro de veinte años.

Un modelo de D+1 **explota la persistencia**. Una curva a largo plazo tiene que ignorarla.
No es la misma herramienta.

## Lo que sí se hace

La descomposición estándar del sector, en tres piezas deliberadamente separadas:

```
precio(dia, hora)  =  nivel(año) x factor_mes  +  forma(mes, tipo_dia, hora)  +  residuo
```

**El nivel no se predice: se aporta.** Sale de los futuros MIBEL —que cotizan a tres o cuatro
años— o de un escenario fundamental. Si no se pasa, el script usa la media de los últimos 12
meses en plano y **avisa de que es un marcador de posición, no una previsión**. Esa separación
es lo que hace defendible el resultado: la parte que se inventa está aislada y etiquetada.

**La forma sí sale de los datos**, y es la parte interesante.

**La banda** sale de remuestrear residuos en bloques de 24 horas.

In [ ]:
import sys
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "gold").is_dir())
sys.path.append(str(REPO / "scripts"))

# Recarga forzada: `curva_precios.py` esta en desarrollo, y Python cachea los modulos
# ya importados. Sin esto, añadir una funcion al script no llega al kernel y salta un
# ImportError que parece un error de nombre y es de cache.
import importlib, curva_precios
importlib.reload(curva_precios)
from curva_precios import (curva, deformacion, historico, perfil,
                           drivers, modelo_spread, por_anclas,
                           dias_molde, simular)

H = historico()
print(f"histórico: {H.dia.min():%Y-%m-%d} -> {H.dia.max():%Y-%m-%d} "
      f"· {H.dia.nunique():,} días · {len(H):,} horas")

## 1 · El perfil se está deformando

Este es el hallazgo que condiciona todo lo demás, y sale directo del histórico.

La tabla mide, para cada año, cuánto se desvía cada hora respecto a **la media de su propio
día** — así se aísla la forma del nivel.

In [ ]:
d = deformacion(H)
display(d)

fig, ax = plt.subplots(1, 2, figsize=(13, 4.2))
ax[0].bar(d.index, d.nivel_medio, color="steelblue")
ax[0].set_title("Nivel medio anual (€/MWh)")
ax[0].set_ylabel("€/MWh"); ax[0].grid(alpha=.3, axis="y")

ax[1].plot(d.index, d.valle_12_15h, "o-", label="valle 12-15 h", color="darkorange")
ax[1].plot(d.index, d.pico_19_21h, "o-", label="pico 19-21 h", color="crimson")
ax[1].fill_between(d.index, d.valle_12_15h, d.pico_19_21h, alpha=.12, color="grey")
ax[1].axhline(0, color="black", lw=.8)
ax[1].set_title("Desviación sobre la media del día (€/MWh)")
ax[1].legend(); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.show()

print(f"el spread intradiario pasa de {d.spread.iloc[0]:.1f} a {d.spread.iloc[-1]:.1f} €/MWh "
      f"-> x{d.spread.iloc[-1]/d.spread.iloc[0]:.0f}")

**Dos cosas que conviene leer juntas.**

El nivel medio lleva plano desde 2024 —63, 65, 66 €/MWh— mientras el spread intradiario **se
ha duplicado**. Son fenómenos independientes: el nivel lo marca el gas y la demanda; el spread
lo marca cuánta solar entra al mediodía.

Y eso es exactamente lo que decide la rentabilidad de una batería. **No gana dinero porque el
precio suba, gana porque el precio de las 14:00 y el de las 20:00 se separen.** El capítulo de
almacenamiento se sostiene sobre esta gráfica, no sobre la de la izquierda.

Por eso el perfil se estima solo con los **últimos dos años**: promediar 2020 con 2026 daría
una forma que no existió nunca y que ya no va a volver.

## 2 · El perfil estimado, hora a hora

Lo que el simulador usa como forma: cuánto se desvía cada hora de la media de su día, por mes
y tipo de día.

In [ ]:
forma, fac_mes, res = perfil(H)
f = forma.reset_index()

fig, ax = plt.subplots(figsize=(10, 4.5))
for mes, col in [(1, "#4a6fa5"), (4, "#5ed69a"), (7, "#fb923c"), (10, "#a78bfa")]:
    s = f[(f.mes == mes) & (f.tipo == "laborable")].set_index("hora").rel
    ax.plot(s.index, s.values, "o-", ms=3, color=col,
            label=["ene", "abr", "jul", "oct"][[1, 4, 7, 10].index(mes)])
ax.axhline(0, color="black", lw=.8)
ax.set_xlabel("hora"); ax.set_ylabel("€/MWh sobre la media del día")
ax.set_title("Perfil intradiario por mes (días laborables, últimos 2 años)")
ax.set_xticks(range(0, 24, 2)); ax.legend(); ax.grid(alpha=.3)
plt.tight_layout(); plt.show()

print("factor estacional por mes (multiplica al nivel anual):")
print(fac_mes.round(3).to_string())

El valle de mediodía es mucho más profundo en **julio** que en enero: más horas de sol y más
producción fotovoltaica. En invierno el perfil casi se aplana y el pico se desplaza.

Eso importa para la batería: **su margen no es constante a lo largo del año**, y un cálculo con
el spread medio anual sobreestima el invierno y subestima el verano.

## 3 · La curva a futuro, con banda

Aquí se pide el rango. El `nivel` es un **escenario que aportas tú** — de los futuros MIBEL o
de un modelo fundamental. Si lo dejas en `None`, el script avisa de que está usando un
marcador de posición.

In [ ]:
# El rango es libre: cambia estas dos fechas y todo lo demas se ajusta.
DESDE, HASTA = "2027-01-01", "2030-12-31"
ESCENARIOS = 400

# Nivel anual en €/MWh, POR ANCLAS. Se dan los años que se saben y el resto se interpola,
# que es lo unico que permite pedir veinte años sin escribir veinte numeros.
#   - los primeros se anclan a los futuros MIBEL
#   - el ultimo es una hipotesis propia, y hay que decirlo en la memoria
ANCLAS_NIVEL = {2027: 66, 2030: 60, 2040: 55, 2046: 52}

a0, a1 = int(DESDE[:4]), int(HASTA[:4])
NIVEL = por_anclas(ANCLAS_NIVEL, a0, a1)
print("nivel anual (€/MWh):", {k: round(v, 1) for k, v in NIVEL.items()})

c = curva(DESDE, HASTA, nivel=NIVEL, n=ESCENARIOS)
print()
display(c.groupby("origen").agg(dias=("dia", "nunique"), media=("p50", "mean")).round(2))

## 4 · Cómo se ve

In [ ]:
s = c[c.origen == "simulado"].copy()
s["mes"] = s.dia.dt.to_period("M").dt.to_timestamp()
m = s.groupby("mes")[["p10", "p50", "p90"]].mean()

fig, ax = plt.subplots(2, 1, figsize=(12, 8))

ax[0].fill_between(m.index, m.p10, m.p90, alpha=.2, color="steelblue", label="P10-P90")
ax[0].plot(m.index, m.p50, color="steelblue", lw=2, label="P50")
hm = H[H.dia > H.dia.max() - pd.DateOffset(years=2)].copy()
hm["mes"] = hm.dia.dt.to_period("M").dt.to_timestamp()
ax[0].plot(hm.groupby("mes").precio.mean(), color="black", lw=1.6, label="histórico")
ax[0].set_ylabel("€/MWh"); ax[0].legend(); ax[0].grid(alpha=.3)
ax[0].set_title("Media mensual: histórico y curva simulada con banda")

sem = s[(s.dia >= "2028-06-05") & (s.dia <= "2028-06-11")]
x = range(len(sem))
ax[1].fill_between(x, sem.p10, sem.p90, alpha=.2, color="crimson")
ax[1].plot(x, sem.p50, color="crimson", lw=1.6)
ax[1].axhline(0, color="black", lw=.8)
ax[1].set_title("Una semana de junio de 2028, hora a hora")
ax[1].set_xlabel("horas"); ax[1].set_ylabel("€/MWh"); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.show()

neg = (s.p10 < 0).mean() * 100
print(f"horas con P10 negativo: {neg:.1f}%  ·  ancho medio de la banda: "
      f"{(s.p90 - s.p10).mean():.1f} €/MWh")

## 5 · Lo que esta curva NO es

Conviene decirlo en la memoria con la misma claridad con la que se presenta el resultado.

**El nivel es un supuesto, no una predicción.** Todo el largo plazo cuelga del escenario que
se introduce en `NIVEL`. Si ese escenario está mal, la curva está mal por mucho que la forma
y la banda sean correctas.

**La banda mide la variabilidad histórica, no la incertidumbre del escenario.** Los
percentiles salen de remuestrear residuos de los últimos dos años: recogen cómo de variable es
el precio *dado* un nivel, no la probabilidad de que el nivel sea otro. La incertidumbre real a
diez años es bastante mayor que esta banda.

**La forma se supone estable, y no lo es.** Se estima con los últimos dos años y se mantiene
fija hacia adelante. Pero la sección 1 demuestra justo lo contrario: el valle se hunde año a
año. Extrapolar esa deriva es la mejora más obvia y es lo primero que haría en una siguiente
versión.

**No hay modelo fundamental detrás.** Un despacho real —PyPSA, Antares, PLEXOS— calcularía el
precio a partir del parque instalado, la demanda y los combustibles. Aquí se toma el nivel como
dato y solo se modela la forma.

## 6 · Proyectar la forma en vez de congelarla

La sección 5 admitía la debilidad más gorda: la forma se estima con los últimos dos años y se
mantiene fija. Pero la sección 1 demuestra que **la forma es justo lo que más se mueve**.

Aquí se arregla. La idea: en vez de extrapolar el precio, se modela **de qué depende la
forma** y se evalúa esa función en escenarios futuros.

La variable candidata es la capacidad solar instalada, que sí tiene proyecciones creíbles —el
PNIEC fija 76 GW de fotovoltaica para 2030— mientras que el precio no las tiene.

In [ ]:
from curva_precios import drivers, modelo_spread

D = drivers()
pred, info = modelo_spread(D)
print(f"{info['meses']} meses · R² = {info['R2']}")
print(f"cada GW de solar abre el spread {info['pendiente_EUR_por_GW']:+.2f} €/MWh")
print(f"correlación con solar {info['corr_solar']}  ·  con eólica {info['corr_eolica']}")

fig, ax = plt.subplots(1, 2, figsize=(13, 4.2))
ax[0].scatter(D.solar_gw, D.spread, s=22, c=D.index.year, cmap="viridis")
gw = np.linspace(D.solar_gw.min(), 120, 50)
ax[0].plot(gw, [pred(g) for g in gw], color="crimson", lw=1.6, label="ajuste lineal")
for g, et in [(76, "PNIEC 2030")]:
    ax[0].axvline(g, ls="--", color="grey", lw=1)
    ax[0].annotate(et, (g, ax[0].get_ylim()[0]), rotation=90, fontsize=8,
                   va="bottom", ha="right", color="grey")
ax[0].set_xlabel("solar instalada (GW)"); ax[0].set_ylabel("spread intradiario (€/MWh)")
ax[0].set_title("El spread crece con el parque solar"); ax[0].legend(); ax[0].grid(alpha=.3)

ax[1].plot(D.index.to_timestamp(), D.solar_gw, color="darkorange", label="solar")
ax[1].plot(D.index.to_timestamp(), D.eolica_gw, color="steelblue", label="eólica")
ax[1].set_ylabel("GW instalados"); ax[1].legend(); ax[1].grid(alpha=.3)
ax[1].set_title("Y las dos crecen a la vez — de ahí la advertencia de abajo")
plt.tight_layout(); plt.show()

**La advertencia, y es importante.** La eólica correlaciona 0,77 con el spread y la solar
0,73: prácticamente igual. Las dos crecen con el tiempo, así que **el tiempo confunde a las
dos** y esto no demuestra que sea la solar la que abre el spread.

Como modelo de primer orden para proyectar, sirve — el mecanismo físico es conocido y la
gráfica de la izquierda es consistente con él. Como afirmación causal, no. En la memoria hay
que presentarlo así: *el spread crece con el parque renovable, y se usa la solar como variable
de proyección porque es la que tiene objetivos publicados*.

## 7 · Veinte años en una celda

Aquí está todo junto: nivel por escenario, forma proyectada según la capacidad solar prevista,
y banda de percentiles. **175.320 horas.**

In [ ]:
# ─── LOS DOS ESCENARIOS QUE HAY QUE APORTAR ────────────────────────────────────
# Nada de esto sale de los datos: son los supuestos sobre los que descansa la curva.
ANO_INI, ANO_FIN = 2027, 2046          # <- pide el rango que quieras

# 1. Nivel anual en €/MWh, por anclas interpoladas
ANCLAS_NIVEL = {2027: 66, 2030: 60, 2035: 57, 2040: 55, 2046: 52}

# 2. Capacidad solar en GW. El PNIEC fija 76 GW para 2030; despues, hipotesis propia.
ANCLAS_SOLAR = {2027: 67, 2030: 76, 2035: 95, 2040: 110, 2046: 125}
# ───────────────────────────────────────────────────────────────────────────────

NIVEL = por_anclas(ANCLAS_NIVEL, ANO_INI, ANO_FIN)
SOLAR = por_anclas(ANCLAS_SOLAR, ANO_INI, ANO_FIN)

C20 = curva(f"{ANO_INI}-01-01", f"{ANO_FIN}-12-31",
            nivel=NIVEL, solar_gw=SOLAR, n=200, verbose=False)

an = C20.assign(y=C20.dia.dt.year).groupby("y").agg(
    p10=("p10", "mean"), p50=("p50", "mean"), p90=("p90", "mean"))
sp = C20.assign(y=C20.dia.dt.year, h=C20.hora)
sp["rel"] = sp.p50 - sp.groupby("dia").p50.transform("mean")
an["valle"] = sp[sp.h.between(12, 15)].groupby("y").rel.mean()
an["pico"] = sp[sp.h.between(19, 21)].groupby("y").rel.mean()
an["spread"] = an.pico - an.valle
an["solar_GW"] = pd.Series(SOLAR)
an["h_negativas_%"] = C20.assign(y=C20.dia.dt.year).groupby("y").p50.apply(
    lambda x: (x < 0).mean() * 100)

fig = plt.figure(figsize=(13, 8.5))
g = fig.add_gridspec(2, 2, height_ratios=[1.25, 1])

a0 = fig.add_subplot(g[0, :])
hh = H.groupby(H.dia.dt.year).precio.mean()
a0.plot(hh.index, hh.values, "o-", color="black", lw=1.8, label="histórico")
a0.fill_between(an.index, an.p10, an.p90, alpha=.2, color="steelblue", label="P10-P90")
a0.plot(an.index, an.p50, color="steelblue", lw=2, label="P50 simulado")
a0.axvline(2026.5, ls="--", color="grey", lw=1)
a0.set_ylabel("€/MWh"); a0.legend(); a0.grid(alpha=.3)
a0.set_title(f"Precio medio anual · histórico {hh.index.min()}-{hh.index.max()} "
             f"y curva {ANO_INI}-{ANO_FIN}")

a1 = fig.add_subplot(g[1, 0])
dd = deformacion(H)
a1.plot(dd.index, dd.spread, "o-", color="black", label="observado")
a1.plot(an.index, an.spread, color="crimson", lw=2, label="proyectado")
a1.set_ylabel("spread intradiario (€/MWh)"); a1.legend(); a1.grid(alpha=.3)
a1.set_title("El spread, que es lo que paga la batería")

a2 = fig.add_subplot(g[1, 1])
tercios = [ANO_INI, (ANO_INI + ANO_FIN) // 2, ANO_FIN]
for y, col in zip(tercios, ["#4a6fa5", "#fb923c", "#c0392b"]):
    q = sp[(sp.y == y) & (sp.dia.dt.month == 7)].groupby("h").rel.mean()
    a2.plot(q.index, q.values, "o-", ms=3, color=col, label=str(y))
a2.axhline(0, color="black", lw=.8)
a2.set_xlabel("hora"); a2.set_ylabel("€/MWh sobre la media del día")
a2.set_title("Perfil de julio, cómo se ahonda"); a2.legend(); a2.grid(alpha=.3)
plt.tight_layout(); plt.show()

print(f"{len(C20):,} horas · {C20.dia.nunique():,} días · "
      f"{ANO_FIN - ANO_INI + 1} años completos\n")
display(an.round(1))

## 8 · La curva horaria

Dibujar las 175.320 horas seguidas da un manchón: veinte años en un eje no dejan ver una
sola hora. Así que va en cuatro imágenes separadas, cada una a su tamaño.

Los tres números de siempre en todas: **mínimo estimado (P10), curva propuesta (P50) y
máximo estimado (P90)**.

In [ ]:
# percentiles consolidados por hora del dia, sobre TODO el periodo
cons = C20.groupby("hora").agg(
    minimo=("p10", "mean"), estimacion=("p50", "mean"), maximo=("p90", "mean")).round(2)
cons["banda"] = (cons.maximo - cons.minimo).round(1)
hoy = H[H.dia > H.dia.max() - pd.DateOffset(years=1)].groupby("hora").precio.mean()

fig, ax = plt.subplots(figsize=(12, 5.5))
ax.fill_between(cons.index, cons.minimo, cons.maximo, alpha=.22, color="steelblue",
                label="rango estimado P10 - P90")
ax.plot(cons.index, cons.minimo, lw=1, color="steelblue", alpha=.7)
ax.plot(cons.index, cons.maximo, lw=1, color="steelblue", alpha=.7)
ax.plot(cons.index, cons.estimacion, "o-", color="#0d47a1", lw=2.6, ms=5,
        label="curva propuesta (P50)")
ax.plot(hoy.index, hoy.values, "--", color="black", lw=1.6, label="últimos 12 meses reales")
ax.axhline(0, color="grey", lw=.9)
ax.set_xticks(range(24)); ax.set_xlabel("hora del día"); ax.set_ylabel("€/MWh")
ax.set_title(f"Curva horaria consolidada {ANO_INI}-{ANO_FIN} · media de {C20.dia.nunique():,} días",
             fontsize=13)
ax.legend(); ax.grid(alpha=.3)
plt.tight_layout(); plt.show()

La curva propuesta frente a lo que pasa hoy: **el valle se hunde y el pico sube**. La banda es
ancha a media tarde —cuando el precio depende de si hay viento o no— y estrecha de madrugada.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5.5))
anos = [ANO_INI, ANO_INI + (ANO_FIN - ANO_INI) // 3,
        ANO_INI + 2 * (ANO_FIN - ANO_INI) // 3, ANO_FIN]
cols = ["#90caf9", "#42a5f5", "#1565c0", "#0d1b4b"]
for y, col in zip(anos, cols):
    q = C20[C20.dia.dt.year == y].groupby("hora").p50.mean()
    ax.plot(q.index, q.values, "o-", ms=4, lw=2, color=col, label=str(y))
ax.plot(hoy.index, hoy.values, "--", color="crimson", lw=2, label="hoy (12 meses reales)")
ax.axhline(0, color="grey", lw=.9)
ax.set_xticks(range(24)); ax.set_xlabel("hora del día"); ax.set_ylabel("€/MWh")
ax.set_title("Cómo evoluciona el perfil: el valle de mediodía se ahonda", fontsize=13)
ax.legend(); ax.grid(alpha=.3)
plt.tight_layout(); plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5.5))
dia = C20.groupby("dia").agg(p10=("p10", "mean"), p50=("p50", "mean"), p90=("p90", "mean"))
ax.fill_between(dia.index, dia.p10, dia.p90, alpha=.18, color="steelblue", lw=0,
                label="rango P10 - P90")
ax.plot(dia.index, dia.p50, color="steelblue", lw=.35, alpha=.8)
ax.plot(dia.index, dia.p50.rolling(90, center=True).mean(), color="#0d47a1", lw=2,
        label="curva propuesta (media móvil 90 d)")
hd = H.groupby("dia").precio.mean()
ax.plot(hd.index, hd.rolling(90, center=True).mean(), color="black", lw=2, label="histórico")
ax.axhline(0, color="grey", lw=.9)
ax.set_ylabel("€/MWh")
ax.set_title(f"Periodo completo · media diaria · {C20.dia.nunique():,} días", fontsize=13)
ax.legend(); ax.grid(alpha=.3)
plt.tight_layout(); plt.show()

In [ ]:
# una semana concreta, hora a hora, para ver el detalle que las otras esconden
SEMANA = f"{(ANO_INI + ANO_FIN) // 2}-07-06"
sem = C20[(C20.dia >= SEMANA) & (C20.dia < pd.Timestamp(SEMANA) + pd.Timedelta(days=7))]
ts = sem.dia + pd.to_timedelta(sem.hora, unit="h")

fig, ax = plt.subplots(figsize=(14, 5))
ax.fill_between(ts, sem.p10, sem.p90, alpha=.25, color="steelblue", label="rango P10 - P90")
ax.plot(ts, sem.p50, color="#0d47a1", lw=1.8, label="curva propuesta (P50)")
ax.axhline(0, color="grey", lw=.9)
for d in pd.date_range(SEMANA, periods=7):
    ax.axvline(d, color="grey", lw=.5, alpha=.5)
ax.set_ylabel("€/MWh")
ax.set_title(f"Detalle horario · semana del {pd.Timestamp(SEMANA):%d-%m-%Y}", fontsize=13)
ax.legend(); ax.grid(alpha=.3)
plt.tight_layout(); plt.show()

print(f"  CURVA HORARIA CONSOLIDADA {ANO_INI}-{ANO_FIN}  ({len(C20):,} horas)")
print(f"  {'hora':>5s} {'minimo':>9s} {'propuesta':>11s} {'maximo':>9s} {'banda':>8s}")
print("  " + "-" * 48)
for h, r in cons.iterrows():
    marca = "  <- valle" if h == cons.estimacion.idxmin() else (
            "  <- pico" if h == cons.estimacion.idxmax() else "")
    print(f"  {h:5d} {r.minimo:9.2f} {r.estimacion:11.2f} {r.maximo:9.2f} "
          f"{r.banda:8.1f}{marca}")
print("  " + "-" * 48)
print(f"  {'media':>5s} {cons.minimo.mean():9.2f} {cons.estimacion.mean():11.2f} "
      f"{cons.maximo.mean():9.2f} {cons.banda.mean():8.1f}")
print()
print(f"  spread de la curva consolidada: "
      f"{cons.estimacion.max() - cons.estimacion.min():.1f} EUR/MWh")
print(f"  el de los ultimos 12 meses reales: {hoy.max() - hoy.min():.1f}")

## 8c · La curva horaria impresa

Los tres números que se piden para cada hora: **mínimo estimado, curva propuesta y máximo
estimado**. P10, P50 y P90.

La `estimacion` es la curva a usar. El `minimo` y el `maximo` no son el peor y el mejor caso
posibles: son los percentiles 10 y 90, o sea que **una de cada cinco horas caerá fuera de esa
banda**. Es deliberado — una banda P0-P100 sería tan ancha que no diría nada.

In [ ]:
# ── cambia esta fecha para imprimir cualquier dia ────────────────────────────
DIA = "2030-07-15"
# ─────────────────────────────────────────────────────────────────────────────

T = C20.rename(columns={"p10": "minimo", "p50": "estimacion", "p90": "maximo"})
T = T[["dia", "hora", "minimo", "estimacion", "maximo"]].copy()
T["banda"] = (T.maximo - T.minimo).round(1)

uno = T[T.dia == DIA].drop(columns="dia").set_index("hora")
print(f"  CURVA HORARIA · {pd.Timestamp(DIA):%d-%m-%Y}")
print(f"  {'hora':>5s} {'minimo':>9s} {'estimacion':>12s} {'maximo':>9s} {'banda':>8s}")
print("  " + "-" * 50)
i_min, i_max = uno.estimacion.idxmin(), uno.estimacion.idxmax()
for h, r in uno.iterrows():
    marca = "  <- valle" if h == i_min else ("  <- pico" if h == i_max else "")
    print(f"  {h:5d} {r.minimo:9.2f} {r.estimacion:12.2f} {r.maximo:9.2f} "
          f"{r.banda:8.1f}{marca}")
print("  " + "-" * 50)
print(f"  {'media':>5s} {uno.minimo.mean():9.2f} {uno.estimacion.mean():12.2f} "
      f"{uno.maximo.mean():9.2f} {uno.banda.mean():8.1f}")
print()
print(f"  spread del dia (pico - valle sobre la estimacion): "
      f"{uno.estimacion.max() - uno.estimacion.min():.1f} EUR/MWh")

print()
print(f"  LA CURVA ENTERA · {len(T):,} horas")
with pd.option_context("display.max_rows", 40, "display.width", 100):
    display(T.set_index(["dia", "hora"]))

Para imprimir **todas** las horas de golpe en vez de las 40 que muestra el resumen:

```python
with pd.option_context("display.max_rows", None):
    display(T.set_index(["dia", "hora"]))
```

Son 175.320 filas, así que el navegador va a sufrir. Para trabajar con ellas de verdad está el
CSV de la sección siguiente.

## 8b · Validación: simular un año que ya conocemos

Una curva a veinte años no se puede validar contra el futuro. Lo que sí se puede es
**esconder un año conocido**: se construye el molde con datos anteriores a 2025, se simula
2025 dándole solo su nivel medio, y se compara con lo que realmente pasó.

Eso es lo único que convierte esta curva en algo defendible en lugar de una gráfica bonita.

In [ ]:
prev = H[H.dia < "2025-01-01"]
molde_bt = dias_molde(prev, anos=2)
_, fac_bt, _ = perfil(prev, anos=2)

bt = simular("2025-01-01", "2025-12-31", {2025: 65.3}, molde_bt, fac_bt,
             n=200, amplitud={2025: 1.0}, percentiles=(1, 10, 50, 90, 99))
j = bt.merge(H[H.dia.dt.year == 2025][["dia", "hora", "precio"]], on=["dia", "hora"])

q = [0, 5, 25, 50, 75, 95, 100]
tab = pd.DataFrame({"real": [np.percentile(j.precio, x) for x in q],
                    "simulado_p50": [np.percentile(j.p50, x) for x in q]},
                   index=[f"p{x}" for x in q]).round(1)
tab["dif"] = (tab.simulado_p50 - tab.real).round(1)
display(tab)

cob = ((j.precio >= j.p1) & (j.precio <= j.p99)).mean() * 100
cob80 = ((j.precio >= j.p10) & (j.precio <= j.p90)).mean() * 100
print(f"media    real {j.precio.mean():.1f}  ·  simulado {j.p50.mean():.1f}")
print(f"h < 0    real {(j.precio<0).mean()*100:.2f}%  ·  simulado {(j.p50<0).mean()*100:.2f}%")
print(f"cobertura P1-P99 {cob:.1f}%  (ideal 98)  ·  P10-P90 {cob80:.1f}%  (ideal 80)")

fig, ax = plt.subplots(figsize=(12, 4))
sem = j[(j.dia >= "2025-06-09") & (j.dia <= "2025-06-15")]
x = range(len(sem))
ax.fill_between(x, sem.p10, sem.p90, alpha=.25, color="steelblue", label="P10-P90")
ax.plot(x, sem.p50, color="steelblue", lw=1.6, label="P50 simulado")
ax.plot(x, sem.precio, color="black", lw=1.4, label="real")
ax.axhline(0, color="grey", lw=.8)
ax.set_title("Una semana de junio de 2025: simulado a ciegas contra lo que ocurrió")
ax.set_xlabel("horas"); ax.set_ylabel("€/MWh"); ax.legend(); ax.grid(alpha=.3)
plt.tight_layout(); plt.show()

**Cómo se lee.** La mediana, el mínimo y el porcentaje de horas negativas encajan bien: 70,1
frente a 70,0, y 5,8 % de horas negativas frente a 6,3 %. La distribución del precio está bien
reproducida.

La cobertura de la banda P1-P99 sale en el 92 % cuando debería rondar el 98: **la banda es algo
más estrecha de lo que debiera**. Hay que decirlo — la incertidumbre real es mayor que la
dibujada.

**Y lo más importante para el capítulo de baterías:** fíjate en que el P50 es más suave que la
línea negra. Es inevitable —una mediana entre 200 escenarios promedia los extremos— pero
significa que **el P50 subestima el spread**. Si calculas el ingreso de una batería sobre la
curva P50, te saldrá menos de lo que ganaría en realidad.

Para el capítulo 7 hay que operar la batería **sobre escenarios individuales** y luego
promediar el ingreso, no operarla sobre el promedio de los escenarios. No es lo mismo, y la
diferencia va en la dirección de infravalorar el negocio.

## 9 · Exportar la curva completa

In [ ]:
# ─── la curva completa a disco ────────────────────────────────────────────────
SALIDA = REPO / "data" / "gold" / f"curva_{ANO_INI}_{ANO_FIN}.csv"

exp = C20.copy()
exp.insert(1, "fecha_hora", exp.dia + pd.to_timedelta(exp.hora, unit="h"))
exp = exp.drop(columns=["dia"])
exp.to_csv(SALIDA, index=False, float_format="%.2f")

print(f"{SALIDA.name}  ·  {len(exp):,} filas  ·  {SALIDA.stat().st_size/2**20:.1f} MB")
print(f"columnas: {list(exp.columns)}\n")
print(exp.head(3).to_string(index=False))
print("   ...")
print(exp.tail(3).to_string(index=False))

# y el resumen anual aparte, que es lo que se pega en la memoria
RES = REPO / "data" / "gold" / f"curva_{ANO_INI}_{ANO_FIN}_anual.csv"
an.round(2).to_csv(RES)
print(f"\nresumen anual en {RES.name}")

**Lo que se ve en la gráfica de abajo a la derecha** es el resultado que importa para el
capítulo de almacenamiento: el valle de julio se hunde año tras año conforme entra solar, y el
pico de la tarde se separa. Esa distancia es literalmente el margen bruto por ciclo de una
batería.

Y conviene mirar la de arriba junto a la de la izquierda: **el precio medio baja mientras el
spread sube.** Un análisis que solo mirara el nivel concluiría que el negocio empeora. Es al
revés.

---

Las cuatro limitaciones de la sección 5 siguen vigentes salvo la tercera, que es la que acaba
de arreglarse: la forma ya no se congela. Pero se proyecta con un ajuste lineal de R² 0,53
sobre variables que el tiempo confunde, así que **la incertidumbre real de esta curva es
bastante mayor que la banda que dibuja**. La banda recoge la variabilidad del precio dado un
escenario; no recoge que el escenario pueda estar equivocado.